# Additional Ontario PHU examples

Calculate trajectories from **existing posterior parameter samples**, then plot fitting and forecasting. No inference is implemented. All inputs and code are inside this extracted package; no precomputed posterior trajectories or verification references are required. Read README.md, then execute cells in order from the package root. All supplied configurations run by default; select one below to shorten execution.

In [ ]:
from pathlib import Path
import sys, time
ROOT = Path.cwd()
assert (ROOT / "posterior_forward.py").is_file(), "Start Jupyter from the extracted package root."
sys.path.insert(0, str(ROOT))
from posterior_forward import case_ids, calculate
print("Available IDs:", case_ids(ROOT))
CASE = "all"              # all, or one of the printed IDs
SAMPLE_FILE = None         # optional compatible CSV / labelled NPZ; one configuration only
CONFIG_FILE = None         # corresponding configuration.json for alternative settings
CONDITIONING = "auto"     # supplied Ottawa mask only with its exact bundled samples
OUTPUT = ROOT / "outputs" / f"forward_{time.time_ns()}"


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Image
from io import BytesIO
selected = case_ids(ROOT) if CASE == "all" else [CASE]
if CONFIG_FILE is not None:
    selected = ["custom"]
if SAMPLE_FILE is not None and len(selected) != 1:
    raise ValueError("Choose a single CASE or CONFIG_FILE for alternative samples.")
results = {}
for case in selected:
    if CONFIG_FILE is None and case not in case_ids(ROOT):
        raise ValueError("Unknown CASE; choose an ID printed above.")
    config = Path(CONFIG_FILE) if CONFIG_FILE is not None else ROOT / "cases" / case / "configuration.json"
    print(f"Calculating {case} from original posterior rows", flush=True)
    info, figure, data = calculate(config, OUTPUT / case, SAMPLE_FILE, CONDITIONING)
    results[case] = {"info": info, "data": data}
    print("Status:", info["status"], "Output:", info["output"])
    print("Daily time (days):", data["time"][0], "through", data["time"][-1])
    print("E/I/beta shapes:", data["E"].shape, data["I"].shape, data["beta"].shape)
    print("Summary samples:", (data["weights"] > 0).sum(), "of", len(data["weights"]), "; weight sum:", data["weights"].sum())
    print("Full-domain invalid original row IDs:", info["invalid_original_rows"])
    if figure is not None:
        preview = BytesIO()
        figure.savefig(preview, format="png", dpi=110, bbox_inches="tight")
        display(Image(data=preview.getvalue()))
        plt.close(figure)


`results[id]["data"]` holds aligned daily arrays of S/E/I/R/D (persons), beta (day^-1), time (days), original and normalized summary weights, row IDs and physical validity. Arrays include every input row. The same summary weights are used for fitting and forecasting. The blue bands exclude observation noise; dashed black curves, where supplied, are generating truth. Original Ottawa rows 443,2837,3048,3113 remain present with zero summary weight under its documented conditional baseline. No historical mask is automatically applied to alternative samples. Raw generated arrays and validity diagnostics remain available if summaries are withheld.